In [30]:
import sqlite3
import pandas as pd
from sqlalchemy import create_engine, MetaData, Table
from loguru import logger

# Maak een logbestand aan. Bewaart INFO, WARNINGS en ERRORS.
logger.add("sdm_etl.log", rotation="1 MB", level="INFO")
logger.info("Stap 1: Libraries geladen.")

2026-03-06 16:26:11.976 | INFO     | __main__:<module>:8 - Stap 1: Libraries geladen.


In [31]:
# Doeldatabase
sdm_db_path = 'SDM.db'
conn_sdm = sqlite3.connect(sdm_db_path)
cursor_sdm = conn_sdm.cursor()

# Operationele brondatabases
conn_acc_verk = sqlite3.connect('BikeToDrive_1_Accessoireverkoop.db')
conn_fts_verk = sqlite3.connect('BikeToDrive_2_Fietsverkoop.db')
conn_ond      = sqlite3.connect('BikeToDrive_3_Onderhoud.db')
conn_acc_ink  = sqlite3.connect('BikeToDrive_4_Accessoire_Inkoop.db')
conn_fts_ink  = sqlite3.connect('BikeToDrive_5_Fiets_Inkoop.db')

# Engine voor de snelle Bulk Load
engine_sdm = create_engine(f"sqlite:///{sdm_db_path}", future=True)

logger.info("Stap 2: Connecties succesvol geopend.")

2026-03-06 16:26:12.026 | INFO     | __main__:<module>:16 - Stap 2: Connecties succesvol geopend.


In [32]:
def reset_sdm():
    logger.info("Start Full Refresh: SDM leegmaken.")
    cursor_sdm.execute("PRAGMA foreign_keys = OFF;")

    # Tabellen in de juiste volgorde (feitentabellen / koppeltabellen eerst, dimensies laatst)
    tabellen = [
        "Accessoire_Inkoop", "Accessoire_Inkoop_Accessoire", "Accessoire_Inkoop_Leverancier",
        "Accessoireverkoop_Accessoire_Verkoop", "Accessoireverkoop_Monteur", "Accessoireverkoop_Accessoire",
        "Accessoireverkoop_Leverancier", "Accessoireverkoop_Klant", "Accessoireverkoop_Filiaal",
        "Onderhoud", "Onderhoud_Monteur", "Onderhoud_Filiaal", "Onderhoud_Fiets", "Onderhoud_Fabrikant",
        "Fiets_Inkoop", "Fiets_Inkoop_Fiets", "Fiets_Inkoop_Fabrikant",
        "Fietsverkoop_Fiets_Verkoop", "Fietsverkoop_Fiets", "Fietsverkoop_Fabrikant",
        "Fietsverkoop_Monteur", "Fietsverkoop_Klant", "Fietsverkoop_Filiaal"
    ]

    for tabel in tabellen:
        cursor_sdm.execute(f"DELETE FROM {tabel};")

    conn_sdm.commit()
    cursor_sdm.execute("PRAGMA foreign_keys = ON;") # Foreign keys weer aanzetten!
    logger.info("SDM is leeg. Klaar voor nieuwe data.")

# Voer de functie direct uit
reset_sdm()

2026-03-06 16:26:12.059 | INFO     | __main__:reset_sdm:2 - Start Full Refresh: SDM leegmaken.
2026-03-06 16:26:12.070 | INFO     | __main__:reset_sdm:21 - SDM is leeg. Klaar voor nieuwe data.


In [33]:
def schema_driven_bulk_load(bron_conn, query, doel_tabel_naam):
    # 1. Haal data uit de bron
    df_bron = pd.read_sql_query(query, bron_conn)
    if df_bron.empty: return

    # 2. FIX: Zet tekst om naar echte Date/Time objecten voor SQLAlchemy
    if 'datum' in df_bron.columns:
        df_bron['datum'] = pd.to_datetime(df_bron['datum']).dt.date
    if 'starttijd' in df_bron.columns:
        df_bron['starttijd'] = pd.to_datetime(df_bron['starttijd']).dt.time
    if 'eindtijd' in df_bron.columns:
        df_bron['eindtijd'] = pd.to_datetime(df_bron['eindtijd']).dt.time
    if 'geboortedatum' in df_bron.columns:
        df_bron['geboortedatum'] = pd.to_datetime(df_bron['geboortedatum']).dt.date

    # 3. Omzetten naar dictionaries en inladen via SQLAlchemy
    records = df_bron.to_dict(orient='records')
    doel_tabel = Table(doel_tabel_naam, MetaData(), autoload_with=engine_sdm)

    try:
        with engine_sdm.begin() as conn:
            conn.execute(doel_tabel.insert(), records)
        logger.info(f"✅ {len(records)} rijen geladen in {doel_tabel_naam}.")
    except Exception as e:
        logger.error(f"❌ Fout bij {doel_tabel_naam}: {e}")

In [34]:
logger.info("Start inladen van alle data...")

# -- 1. Accessoireverkoop --
schema_driven_bulk_load(conn_acc_verk, "SELECT leveranciernr, naam, adres, woonplaats FROM Leverancier", "Accessoireverkoop_Leverancier")
schema_driven_bulk_load(conn_acc_verk, "SELECT klantnr, naam, adres, woonplaats, geslacht, geboortedatum FROM Klant", "Accessoireverkoop_Klant")
schema_driven_bulk_load(conn_acc_verk, "SELECT filiaalnr, naam, adres, provincie FROM Filiaal", "Accessoireverkoop_Filiaal")
schema_driven_bulk_load(conn_acc_verk, "SELECT accessoirenr, naam, standaardprijs, inkoopprijs, soort, leverancier FROM Accessoire", "Accessoireverkoop_Accessoire")
schema_driven_bulk_load(conn_acc_verk, "SELECT monteurnr, naam, woonplaats, uurloon, filiaal FROM Monteur", "Accessoireverkoop_Monteur")
schema_driven_bulk_load(conn_acc_verk, "SELECT accessoire_verkoopnr, datum, aantal, verkoopprijs, klant, accessoire, monteur FROM Accessoire_Verkoop", "Accessoireverkoop_Accessoire_Verkoop")

# -- 2. Fietsverkoop --
schema_driven_bulk_load(conn_fts_verk, "SELECT fabrikantnr, naam, adres, plaats FROM Fabrikant", "Fietsverkoop_Fabrikant")
schema_driven_bulk_load(conn_fts_verk, "SELECT filiaalnr, naam, adres, provincie FROM Filiaal", "Fietsverkoop_Filiaal")
schema_driven_bulk_load(conn_fts_verk, "SELECT klantnr, naam, adres, woonplaats, geslacht, geboortedatum FROM Klant", "Fietsverkoop_Klant")
schema_driven_bulk_load(conn_fts_verk, "SELECT fietsnr, soort, merk, type, standaardprijs, inkoopprijs, kleur, fabrikant FROM Fiets", "Fietsverkoop_Fiets")
schema_driven_bulk_load(conn_fts_verk, "SELECT monteurnr, naam, woonplaats, uurloon, filiaal FROM Monteur", "Fietsverkoop_Monteur")
schema_driven_bulk_load(conn_fts_verk, "SELECT fiets_verkoopnr, datum, aantal, verkoopprijs, klant, fiets, monteur FROM Fiets_Verkoop", "Fietsverkoop_Fiets_Verkoop")

# -- 3. Onderhoud --
schema_driven_bulk_load(conn_ond, "SELECT fabrikantnr, naam, adres, plaats FROM Fabrikant", "Onderhoud_Fabrikant")
schema_driven_bulk_load(conn_ond, "SELECT filiaalnr, naam, adres, provincie FROM Filiaal", "Onderhoud_Filiaal")
schema_driven_bulk_load(conn_ond, "SELECT fietsnr, soort, merk, type, standaardprijs, inkoopprijs, kleur, fabrikant FROM Fiets", "Onderhoud_Fiets")
schema_driven_bulk_load(conn_ond, "SELECT monteurnr, naam, woonplaats, uurloon, filiaal FROM Monteur", "Onderhoud_Monteur")
schema_driven_bulk_load(conn_ond, "SELECT onderhoudnr, datum, starttijd, eindtijd, fiets, monteur FROM Onderhoud", "Onderhoud")

# -- 4. Accessoire Inkoop --
schema_driven_bulk_load(conn_acc_ink, "SELECT leveranciernr, naam, adres, woonplaats FROM Leverancier", "Accessoire_Inkoop_Leverancier")
schema_driven_bulk_load(conn_acc_ink, "SELECT accessoirenr, naam, standaardprijs, inkoopprijs, soort, leverancier FROM Accessoire", "Accessoire_Inkoop_Accessoire")
schema_driven_bulk_load(conn_acc_ink, "SELECT inkoopnr, inkoopmaand, inkoopjaar, aantal, accessoire FROM Accessoire_Inkoop", "Accessoire_Inkoop")

# -- 5. Fiets Inkoop --
schema_driven_bulk_load(conn_fts_ink, "SELECT fabrikantnr, naam, adres, plaats FROM Fabrikant", "Fiets_Inkoop_Fabrikant")
schema_driven_bulk_load(conn_fts_ink, "SELECT fietsnr, soort, merk, type, standaardprijs, inkoopprijs, kleur, fabrikant FROM Fiets", "Fiets_Inkoop_Fiets")
schema_driven_bulk_load(conn_fts_ink, "SELECT inkoopnr, inkoopmaand, inkoopjaar, aantal, fiets FROM Fiets_Inkoop", "Fiets_Inkoop")

logger.info("🎉 Klaar met inladen!")

2026-03-06 16:26:12.146 | INFO     | __main__:<module>:1 - Start inladen van alle data...
2026-03-06 16:26:12.170 | INFO     | __main__:schema_driven_bulk_load:23 - ✅ 5 rijen geladen in Accessoireverkoop_Leverancier.
2026-03-06 16:26:12.187 | INFO     | __main__:schema_driven_bulk_load:23 - ✅ 20 rijen geladen in Accessoireverkoop_Klant.
2026-03-06 16:26:12.201 | INFO     | __main__:schema_driven_bulk_load:23 - ✅ 4 rijen geladen in Accessoireverkoop_Filiaal.
2026-03-06 16:26:12.220 | INFO     | __main__:schema_driven_bulk_load:23 - ✅ 10 rijen geladen in Accessoireverkoop_Accessoire.
2026-03-06 16:26:12.237 | INFO     | __main__:schema_driven_bulk_load:23 - ✅ 10 rijen geladen in Accessoireverkoop_Monteur.
2026-03-06 16:26:12.274 | INFO     | __main__:schema_driven_bulk_load:23 - ✅ 100 rijen geladen in Accessoireverkoop_Accessoire_Verkoop.
2026-03-06 16:26:12.289 | INFO     | __main__:schema_driven_bulk_load:23 - ✅ 10 rijen geladen in Fietsverkoop_Fabrikant.
2026-03-06 16:26:12.303 | INFO

In [35]:
def incremental_load(bron_conn, query, sdm_tabel_naam, primary_key):
    # Haal beide datasets op
    df_bron = pd.read_sql_query(query, bron_conn)
    df_sdm = pd.read_sql_query(f"SELECT * FROM {sdm_tabel_naam}", conn_sdm)

    # Vergelijk. 'left_only' betekent: wel in bron, niet in SDM.
    diff = pd.merge(df_bron, df_sdm, on=primary_key, how='outer', indicator=True, suffixes=('_bron', '_sdm'))
    nieuwe_rijen = diff[diff['_merge'] == 'left_only']

    if nieuwe_rijen.empty:
        logger.info(f"Geen updates voor {sdm_tabel_naam}.")
        return

    # Ruim DataFrame op (verwijder kolommen uit SDM en de indicator)
    kolommen = [c for c in nieuwe_rijen.columns if not c.endswith('_sdm') and c != '_merge']
    df_klaar = nieuwe_rijen[kolommen].rename(columns=lambda x: x.replace('_bron', ''))

    # FIX voor Datums
    if 'datum' in df_klaar.columns: df_klaar['datum'] = pd.to_datetime(df_klaar['datum']).dt.date
    if 'starttijd' in df_klaar.columns: df_klaar['starttijd'] = pd.to_datetime(df_klaar['starttijd']).dt.time
    if 'eindtijd' in df_klaar.columns: df_klaar['eindtijd'] = pd.to_datetime(df_klaar['eindtijd']).dt.time
    if 'geboortedatum' in df_klaar.columns: df_klaar['geboortedatum'] = pd.to_datetime(df_klaar['geboortedatum']).dt.date

    # Inladen naar SDM
    records = df_klaar.to_dict(orient='records')
    doel_tabel = Table(sdm_tabel_naam, MetaData(), autoload_with=engine_sdm)

    with engine_sdm.begin() as conn:
        conn.execute(doel_tabel.insert(), records)
    logger.info(f"✅ {len(records)} NIEUWE rijen toegevoegd aan {sdm_tabel_naam}.")

In [36]:
incremental_load(
    bron_conn=conn_acc_ink,
    query="SELECT leveranciernr, naam, adres, woonplaats FROM Leverancier",
    sdm_tabel_naam="Accessoire_Inkoop_Leverancier",
    primary_key="leveranciernr"
)

2026-03-06 16:26:12.621 | INFO     | __main__:incremental_load:11 - Geen updates voor Accessoire_Inkoop_Leverancier.


In [37]:
# Alles netjes dichtdoen
conn_acc_verk.close()
conn_fts_verk.close()
conn_ond.close()
conn_acc_ink.close()
conn_fts_ink.close()
conn_sdm.close()

logger.info("ETL proces afgerond, alle connecties gesloten.")

2026-03-06 16:26:12.688 | INFO     | __main__:<module>:9 - ETL proces afgerond, alle connecties gesloten.
